# F02-P2 Threat

**Ecosystem Threat: Dryland Forest, Mangrove, Peatland**

Showcasing the disturbance in three different ecosystems, mainly on forest disturbances but not limited to hydrological disturbances for peatland ecosystems. F02-P3 Threat answers “where is the most disturbed area and its drivers”.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: All ecosystems section.

**Known limit of that scope.** Forest disturbance is an internal agreement term for forest degradation with the structural change across 10 years. However, the 10 years of forest degradation is too long, the next v3.1 this will be revised into annual or maximum 5 years analysis.

## Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio

from pyproj import Geod
from rasterio.mask import raster_geometry_mask
from rasterio.mask import mask

import geopandas as gpd
import numpy as np
import rasterio

---
## 3.1 All Ecosystem (Overview)

Reports the total ecosystem and disturbed area across three different ecosystem in hectare.

**Data.** `forest_disturbance_v3.tif: any pixel > 0 = disturbed` following C. Bourgoin, 2024. The 
methodology published following JRC-TMF data on degraded and undisturbed forest. Scene approach
focusses on structural decline on forest using TCC and TCH by SIGnal forest cover. However, the drivers
of degradation only limited to selective logging and forest fire

**Calibration warning.** The 0 to 3 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published JRC-TMF. 
**Decisions locked.**

Structural decline within retained forest, assessed as canopy height deficit
relative to an undisturbed reference population defined at >=120 m from any
disturbed forest. This addresses the growing stock and biomass marker of FAO
(2011, FRA Working Paper 177). It is not a complete assessment of forest
degradation as defined by FAO, and no globally agreed operational definition
currently exists. Detection threshold set at the 5th percentile of reference
population height change, giving a nominal 5% false-positive rate. Results
are reported as area statistics by stratum; per-pixel interpretation is not
supported at 30 m given a product RMSE of 6.6-9.1 m.

**Example render**

## All Ecosystem Screening

| Summary | Value |
|---|---:|
| **Total ecosystem area** | **81,880.23 ha** |
| **Total disturbed area** | **7,383.64 ha (9.02%)** |

### Ecosystem Breakdown

| Ecosystem | Area (ha) | % Total | Disturbed (ha) | Disturbed % |
|---|---:|---:|---:|---:|
| **Dryland forest** | 81,880.23 | 100.00% | 7,383.64 | 9.02% |
| **Mangrove** | 0.00 | 0.00% | 0.00 | 0.00% |
| **Peatland** | 0.00 | 0.00% | 0.00 | 0.00% |
| **Other** | 0.00 | 0.00% | 0.00 | 0.00% |

In [ ]:
def pixel_area_by_row(transform, height):
    """
    Calculate pixel area in hectares by raster row.
    Suitable for EPSG:4326.
    """

    pixel_width = abs(transform.a)

    row_areas = np.zeros(height)

    for row in range(height):

        north = transform.f + row * transform.e
        south = north + transform.e

        west = transform.c
        east = west + pixel_width

        area_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        row_areas[row] = abs(area_m2) / 10000

    return row_areas


def calculate_mask_area_ha(binary_mask, transform):
    """
    Calculate area of True pixels in hectares.
    """

    row_areas = pixel_area_by_row(
        transform,
        binary_mask.shape[0]
    )

    pixels_per_row = binary_mask.sum(axis=1)

    return float(
        np.sum(
            pixels_per_row * row_areas
        )
    )


def read_masked_raster(
    raster_path,
    aoi_geometry,
    aoi_crs
):
    """
    Read only the AOI window from a raster.
    """

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        if src.crs != aoi_crs:

            raster_aoi = (
                gpd.GeoSeries(
                    [aoi_geometry],
                    crs=aoi_crs
                )
                .to_crs(src.crs)
                .iloc[0]
            )

        else:

            raster_aoi = aoi_geometry


        outside_mask, transform, window = (
            raster_geometry_mask(
                src,
                [raster_aoi.__geo_interface__],
                crop=True,
                all_touched=False
            )
        )


        data = src.read(
            1,
            window=window,
            masked=True
        )


        valid_mask = (
            ~outside_mask
            & ~np.ma.getmaskarray(data)
        )


        return (
            data.data,
            valid_mask,
            transform,
            src.crs
        )


# =============================================================================
# Main Analysis
# =============================================================================

def analyze_all_ecosystem():

    # -------------------------------------------------------------------------
    # 1. Read AOI
    # -------------------------------------------------------------------------

    aoi = gpd.read_file(
        USER_AOI
    )

    if aoi.empty:
        raise ValueError(
            "AOI contains no features."
        )

    if aoi.crs is None:
        raise ValueError(
            "AOI has no CRS."
        )


    valid_geometry = aoi.geometry[
        aoi.geometry.notna()
        & ~aoi.geometry.is_empty
    ]


    if valid_geometry.empty:
        raise ValueError(
            "AOI contains no valid geometry."
        )


    aoi_geometry = (
        valid_geometry.union_all()
    )


    # -------------------------------------------------------------------------
    # 2. Read Ecosystem Raster
    # -------------------------------------------------------------------------

    (
        ecosystem_data,
        ecosystem_valid,
        ecosystem_transform,
        ecosystem_crs
    ) = read_masked_raster(
        ECOSYSTEM_RASTER,
        aoi_geometry,
        aoi.crs
    )


    # -------------------------------------------------------------------------
    # 3. Read Disturbance Raster
    # -------------------------------------------------------------------------

    (
        disturbance_data,
        disturbance_valid,
        disturbance_transform,
        disturbance_crs
    ) = read_masked_raster(
        DISTURBANCE_RASTER,
        aoi_geometry,
        aoi.crs
    )


    # -------------------------------------------------------------------------
    # IMPORTANT:
    # Both rasters must share the same clipped grid for direct boolean masking.
    # -------------------------------------------------------------------------

    if ecosystem_data.shape != disturbance_data.shape:

        raise ValueError(
            "Ecosystem and disturbance raster grids do not match "
            "after clipping."
        )


    if not np.allclose(
        ecosystem_transform,
        disturbance_transform
    ):

        raise ValueError(
            "Ecosystem and disturbance raster transforms do not match."
        )


    # -------------------------------------------------------------------------
    # 4. Total Ecosystem Mask
    # -------------------------------------------------------------------------

    total_ecosystem_mask = (
        ecosystem_valid
        & np.isin(
            ecosystem_data,
            list(
                ECOSYSTEM_CLASSES.keys()
            )
        )
    )


    total_ecosystem_area_ha = (
        calculate_mask_area_ha(
            total_ecosystem_mask,
            ecosystem_transform
        )
    )


    # -------------------------------------------------------------------------
    # 5. Total Disturbed Area
    # -------------------------------------------------------------------------

    disturbance_mask = (
        disturbance_valid
        & (disturbance_data > 0)
    )


    total_disturbed_mask = (
        total_ecosystem_mask
        & disturbance_mask
    )


    total_disturbed_area_ha = (
        calculate_mask_area_ha(
            total_disturbed_mask,
            ecosystem_transform
        )
    )


    total_disturbed_percentage = (
        total_disturbed_area_ha
        / total_ecosystem_area_ha
        * 100
        if total_ecosystem_area_ha > 0
        else 0
    )


    # -------------------------------------------------------------------------
    # 6. Ecosystem Breakdown
    # -------------------------------------------------------------------------

    ecosystem_results = {}


    for class_value, class_name in (
        ECOSYSTEM_CLASSES.items()
    ):

        ecosystem_mask = (
            ecosystem_valid
            & (ecosystem_data == class_value)
        )


        ecosystem_area_ha = (
            calculate_mask_area_ha(
                ecosystem_mask,
                ecosystem_transform
            )
        )


        ecosystem_percentage = (
            ecosystem_area_ha
            / total_ecosystem_area_ha
            * 100
            if total_ecosystem_area_ha > 0
            else 0
        )


        # ---------------------------------------------------------------------
        # Disturbed within ecosystem
        # ---------------------------------------------------------------------

        ecosystem_disturbed_mask = (
            ecosystem_mask
            & disturbance_mask
        )


        ecosystem_disturbed_area_ha = (
            calculate_mask_area_ha(
                ecosystem_disturbed_mask,
                ecosystem_transform
            )
        )


        ecosystem_disturbed_percentage = (
            ecosystem_disturbed_area_ha
            / ecosystem_area_ha
            * 100
            if ecosystem_area_ha > 0
            else 0
        )


        ecosystem_results[
            class_name
        ] = {

            "area_ha":
                round(
                    ecosystem_area_ha,
                    2
                ),

            "percentage_total":
                round(
                    ecosystem_percentage,
                    2
                ),

            "disturbed_area_ha":
                round(
                    ecosystem_disturbed_area_ha,
                    2
                ),

            "disturbed_percentage":
                round(
                    ecosystem_disturbed_percentage,
                    2
                )
        }


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "total_ecosystem_area_ha":
            round(
                total_ecosystem_area_ha,
                2
            ),

        "total_disturbed_area_ha":
            round(
                total_disturbed_area_ha,
                2
            ),

        "total_disturbed_percentage":
            round(
                total_disturbed_percentage,
                2
            ),

        "ecosystems":
            ecosystem_results
    }


# =============================================================================
# Run
# =============================================================================

result = analyze_all_ecosystem()


# =============================================================================
# Display
# =============================================================================

print()
print("=" * 70)
print("ALL ECOSYSTEM SCREENING")
print("=" * 70)

print(
    f"Total ecosystem area : "
    f"{result['total_ecosystem_area_ha']:,.2f} ha"
)

print(
    f"Total disturbed area : "
    f"{result['total_disturbed_area_ha']:,.2f} ha "
    f"({result['total_disturbed_percentage']:.2f}%)"
)


print()
print("Ecosystem breakdown:")
print("-" * 90)

print(
    f"{'Ecosystem':<20}"
    f"{'Area (ha)':>15}"
    f"{'% Total':>12}"
    f"{'Disturbed (ha)':>20}"
    f"{'Disturbed %':>15}"
)

print("-" * 90)


for ecosystem_name, values in (
    result["ecosystems"].items()
):

    print(
        f"{ecosystem_name:<20}"
        f"{values['area_ha']:>15,.2f}"
        f"{values['percentage_total']:>11.2f}%"
        f"{values['disturbed_area_ha']:>20,.2f}"
        f"{values['disturbed_percentage']:>14.2f}%"
    )

---
## 3.2 Dryland Forest Ecosystem

Reports the drivers of dryland forest disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.** Post-processing steps identified where fire coincided with the clearing of agricultural land, based on the presence of VIIRS fire alerts (within a 500 m buffer around the alert) and a low post-disturbance normalized-burn ration in the following month’s Sentinen-2 composite.
Post-processing steps masking out the area outside forest_disturbance_v3.tif and re-calibrate
with disaster risk from ADPC. However, this data only consider high and very high disaster risk
as part of forest disturbance.

In [ ]:
# =============================================================================
# Read AOI
# =============================================================================

aoi = gpd.read_file(AOI)

geometry = [
    geom.__geo_interface__
    for geom in aoi.geometry
]


# =============================================================================
# Read raster clipped to AOI
# =============================================================================

def read(path):

    with rasterio.open(path) as src:

        shapes = geometry

        if aoi.crs != src.crs:

            projected = aoi.to_crs(src.crs)

            shapes = [
                geom.__geo_interface__
                for geom in projected.geometry
            ]

        data, transform = mask(
            src,
            shapes,
            crop=True,
            filled=False
        )

        return data[0], transform


# =============================================================================
# Pixel area for EPSG:4326
# =============================================================================

GEOD = Geod(ellps="WGS84")


def area_ha(mask_array, transform):

    total = 0

    for row in range(mask_array.shape[0]):

        count = mask_array[row].sum()

        if count == 0:
            continue

        north = transform.f + row * transform.e
        south = north + transform.e

        west = transform.c
        east = west + transform.a

        pixel_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        total += count * abs(pixel_m2)

    return total / 10000


# =============================================================================
# Load core rasters
# =============================================================================

eco, transform = read(ECOSYSTEM_RASTER)
historical, _ = read(HISTORICAL_RASTER)
forest2024, _ = read(FOREST_2024_RASTER)
disturbance, _ = read(DISTURBANCE_RASTER)
gain, _ = read(FOREST_GAIN_RASTER)
drivers, _ = read(FOREST_DRIVERS_RASTER)


# =============================================================================
# Core masks
# =============================================================================

dryland = eco == DRYLAND

remaining = (
    dryland
    & (historical == REMAINING_FOREST)
)

forest_loss = (
    dryland
    & (historical == FOREST_LOSS)
)

current_forest = (
    dryland
    & (forest2024 == CURRENT_FOREST)
)

disturbed = (
    current_forest
    & (disturbance > 0)
)

forest_gain = (
    dryland
    & (gain == FOREST_GAIN_VALUE)
)


# =============================================================================
# Areas
# =============================================================================

total_area = area_ha(
    dryland,
    transform
)

remaining_area = area_ha(
    remaining,
    transform
)

disturbed_area = area_ha(
    disturbed,
    transform
)

loss_area = area_ha(
    forest_loss,
    transform
)

gain_area = area_ha(
    forest_gain,
    transform
)


# =============================================================================
# Forest drivers
# =============================================================================

driver_values = np.unique(
    drivers[disturbed]
)

non_natural = []
other = []


for value in driver_values:

    value = int(value)

    if value not in FOREST_DRIVER_CLASSES:
        continue

    name = FOREST_DRIVER_CLASSES[value]

    if name == "Non-productive conversion":
        other.append(name)

    else:
        non_natural.append(name)


# =============================================================================
# Natural drivers
# =============================================================================

natural = []

natural_presence = np.zeros(
    disturbed.shape,
    dtype=bool
)


for name, raster_path in NATURAL_DRIVERS.items():

    risk, _ = read(
        raster_path
    )

    risk_present = (
        disturbed
        & (risk == HIGH_RISK)
    )

    if np.any(risk_present):

        natural.append(name)

        natural_presence |= risk_present


# =============================================================================
# Unknown
# =============================================================================

known_forest_driver = (
    disturbed
    & np.isin(
        drivers,
        list(
            FOREST_DRIVER_CLASSES.keys()
        )
    )
)

unknown = (
    disturbed
    & ~known_forest_driver
    & ~natural_presence
)

if np.any(unknown):
    other.append("Unknown")


# =============================================================================
# Output
# =============================================================================

print()
print("DRYLAND FOREST")
print("-" * 70)

print(
    f"Total area       : "
    f"{total_area:,.2f} ha"
)

print(
    f"Remaining forest : "
    f"{remaining_area:,.2f} ha "
    f"({remaining_area / total_area * 100:.2f}%)"
)

print(
    f"Disturbed        : "
    f"{disturbed_area:,.2f} ha "
    f"({disturbed_area / total_area * 100:.2f}%)"
)

print(
    f"Forest loss      : "
    f"{loss_area:,.2f} ha "
    f"({loss_area / total_area * 100:.2f}%)"
)

print(
    f"Forest gain      : "
    f"{gain_area:,.2f} ha "
    f"({gain_area / total_area * 100:.2f}%)"
)


print()
print("Non-natural drivers:")
for value in non_natural:
    print(f"  - {value}")


print()
print("Natural drivers:")
for value in natural:
    print(f"  - {value}")


print()
print("Other drivers:")
for value in other:
    print(f"  - {value}")

---
## 3.3 Mangrove Forest Ecosystem

Reports the drivers of dryland forest disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.** Post-processing steps identified where fire coincided with the clearing of agricultural land, based on the presence of VIIRS fire alerts (within a 500 m buffer around the alert) and a low post-disturbance normalized-burn ration in the following month’s Sentinen-2 composite.
Post-processing steps masking out the area outside forest_disturbance_v3.tif and re-calibrate
with disaster risk from ADPC. However, this data only consider high and very high disaster risk
as part of forest disturbance.

In [ ]:
aoi = gpd.read_file(
    USER_AOI
)

if aoi.empty:
    raise ValueError(
        "AOI contains no features."
    )

if aoi.crs is None:
    raise ValueError(
        "AOI has no CRS."
    )


aoi = aoi[
    aoi.geometry.notna()
    & ~aoi.geometry.is_empty
].copy()

if aoi.empty:
    raise ValueError(
        "AOI contains no valid geometry."
    )


# =============================================================================
# Helper: clip raster to AOI
# =============================================================================

def read_raster(raster_path):

    with rasterio.open(
        raster_path
    ) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        polygon = (
            aoi.to_crs(src.crs)
            if aoi.crs != src.crs
            else aoi
        )

        geometries = [
            geom.__geo_interface__
            for geom in polygon.geometry
        ]

        data, transform = mask(
            src,
            geometries,
            crop=True,
            filled=False
        )

        return (
            data[0],
            transform,
            src.crs
        )


# =============================================================================
# Helper: check raster grid
# =============================================================================

def check_grid(
    reference,
    reference_transform,
    data,
    transform,
    raster_name
):

    if reference.shape != data.shape:
        raise ValueError(
            f"Grid size mismatch: {raster_name}"
        )

    if not np.allclose(
        reference_transform,
        transform
    ):
        raise ValueError(
            f"Grid alignment mismatch: {raster_name}"
        )


# =============================================================================
# Helper: calculate area in hectares
# =============================================================================

def area_ha(
    binary_mask,
    transform
):
    """
    Geodesic pixel area calculation for EPSG:4326.
    """

    total_m2 = 0.0

    pixel_width = abs(
        transform.a
    )

    for row in range(
        binary_mask.shape[0]
    ):

        pixel_count = int(
            binary_mask[row].sum()
        )

        if pixel_count == 0:
            continue

        north = (
            transform.f
            + row * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        pixel_area_m2, _ = (
            GEOD.polygon_area_perimeter(
                [west, east, east, west],
                [north, north, south, south]
            )
        )

        total_m2 += (
            pixel_count
            * abs(pixel_area_m2)
        )

    return (
        total_m2 / 10000
    )


# =============================================================================
# Load rasters
# =============================================================================

ecosystem, transform, crs = (
    read_raster(
        ECOSYSTEM_RASTER
    )
)

historical, historical_transform, _ = (
    read_raster(
        HISTORICAL_RASTER
    )
)

forest2024, forest2024_transform, _ = (
    read_raster(
        FOREST_2024_RASTER
    )
)

disturbance, disturbance_transform, _ = (
    read_raster(
        DISTURBANCE_RASTER
    )
)

drivers, drivers_transform, _ = (
    read_raster(
        FOREST_DRIVERS_RASTER
    )
)

storm_surge, storm_transform, _ = (
    read_raster(
        STORM_SURGE_RASTER
    )
)


# =============================================================================
# Validate grids
# =============================================================================

check_grid(
    ecosystem,
    transform,
    historical,
    historical_transform,
    "historical_deforestation_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    forest2024,
    forest2024_transform,
    "forest_2024_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    disturbance,
    disturbance_transform,
    "forest_disturbance_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    drivers,
    drivers_transform,
    "forest_drivers_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    storm_surge,
    storm_transform,
    "storm_surge_v3.tif"
)


# =============================================================================
# Valid pixel masks
# =============================================================================

ecosystem_valid = (
    ~np.ma.getmaskarray(
        ecosystem
    )
)

historical_valid = (
    ~np.ma.getmaskarray(
        historical
    )
)

forest2024_valid = (
    ~np.ma.getmaskarray(
        forest2024
    )
)

disturbance_valid = (
    ~np.ma.getmaskarray(
        disturbance
    )
)

drivers_valid = (
    ~np.ma.getmaskarray(
        drivers
    )
)

storm_valid = (
    ~np.ma.getmaskarray(
        storm_surge
    )
)


# =============================================================================
# 1. Total mangrove
# =============================================================================

mangrove_mask = (
    ecosystem_valid
    & (
        ecosystem.data
        == MANGROVE_CLASS
    )
)

total_mangrove_area = area_ha(
    mangrove_mask,
    transform
)


# =============================================================================
# 2. Remaining mangrove forest
# =============================================================================

remaining_mask = (
    mangrove_mask
    & historical_valid
    & (
        historical.data
        == REMAINING_FOREST_CLASS
    )
)

remaining_area = area_ha(
    remaining_mask,
    transform
)


# =============================================================================
# 3. Disturbed mangrove
# =============================================================================

current_mangrove_mask = (
    mangrove_mask
    & forest2024_valid
    & (
        forest2024.data
        == CURRENT_FOREST_CLASS
    )
)

disturbed_mask = (
    current_mangrove_mask
    & disturbance_valid
    & (
        disturbance.data
        > DISTURBANCE_THRESHOLD
    )
)

disturbed_area = area_ha(
    disturbed_mask,
    transform
)


disturbed_percentage = (
    disturbed_area
    / total_mangrove_area
    * 100
    if total_mangrove_area > 0
    else 0
)


remaining_percentage = (
    remaining_area
    / total_mangrove_area
    * 100
    if total_mangrove_area > 0
    else 0
)


# =============================================================================
# 4. Non-natural drivers
# =============================================================================

commodities_mask = (
    disturbed_mask
    & drivers_valid
    & np.isin(
        drivers.data,
        COMMODITY_CLASSES
    )
)

settlement_mask = (
    disturbed_mask
    & drivers_valid
    & (
        drivers.data
        == SETTLEMENT_CLASS
    )
)


commodities_area = area_ha(
    commodities_mask,
    transform
)

settlement_area = area_ha(
    settlement_mask,
    transform
)


non_natural_drivers = []

if np.any(
    commodities_mask
):
    non_natural_drivers.append(
        "Commodities"
    )

if np.any(
    settlement_mask
):
    non_natural_drivers.append(
        "Settlement"
    )


# =============================================================================
# 5. Main pressure
# =============================================================================

if commodities_area > settlement_area:

    main_pressure = (
        "Commodities"
    )

elif settlement_area > commodities_area:

    main_pressure = (
        "Settlement"
    )

elif (
    commodities_area > 0
    and settlement_area > 0
):

    main_pressure = (
        "Commodities and Settlement"
    )

else:

    main_pressure = (
        "Not identified"
    )


# =============================================================================
# 6. Natural driver
# =============================================================================

storm_surge_mask = (
    disturbed_mask
    & storm_valid
    & np.isin(
        storm_surge.data,
        STORM_SURGE_CLASSES
    )
)


natural_drivers = []

if np.any(
    storm_surge_mask
):

    natural_drivers.append(
        "Extreme climate event"
    )


# =============================================================================
# 7. Other driver
# =============================================================================

known_forest_driver_mask = (
    commodities_mask
    | settlement_mask
)

known_driver_mask = (
    known_forest_driver_mask
    | storm_surge_mask
)


other_mask = (
    disturbed_mask
    & ~known_driver_mask
)


other_drivers = []

if np.any(
    other_mask
):

    other_drivers.append(
        "Other"
    )


# =============================================================================
# Backend result
# =============================================================================

result = {

    "mangrove": {

        "total_area_ha":
            round(
                total_mangrove_area,
                2
            ),

        "remaining_forest": {

            "area_ha":
                round(
                    remaining_area,
                    2
                ),

            "percentage":
                round(
                    remaining_percentage,
                    2
                )
        },

        "disturbed": {

            "area_ha":
                round(
                    disturbed_area,
                    2
                ),

            "percentage":
                round(
                    disturbed_percentage,
                    2
                )
        },

        "main_pressure":
            main_pressure,

        "drivers": {

            "non_natural":
                non_natural_drivers,

            "natural":
                natural_drivers,

            "other":
                other_drivers
        }
    }
}


# =============================================================================
# Display
# =============================================================================

print()
print("=" * 75)
print("MANGROVE DISTURBANCE")
print("=" * 75)

print(
    f"Total area                : "
    f"{result['mangrove']['total_area_ha']:,.2f} ha"
)

print(
    f"Remaining mangrove forest : "
    f"{result['mangrove']['remaining_forest']['area_ha']:,.2f} ha "
    f"({result['mangrove']['remaining_forest']['percentage']:.2f}%)"
)

print(
    f"Disturbed                 : "
    f"{result['mangrove']['disturbed']['area_ha']:,.2f} ha "
    f"({result['mangrove']['disturbed']['percentage']:.2f}%)"
)

print(
    f"Main pressure             : "
    f"{result['mangrove']['main_pressure']}"
)


print()
print("Mangrove disturbance drivers")
print("-" * 75)


print()
print(
    "Non-natural drivers:"
)

if non_natural_drivers:

    for driver in non_natural_drivers:
        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print(
    "Natural drivers:"
)

if natural_drivers:

    for driver in natural_drivers:
        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print(
    "Other drivers:"
)

if other_drivers:

    for driver in other_drivers:
        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )